# Uda-hub CultPass Multi-Agent Support Application Notebook

Demonstrates end-to-end ticket processing workflow across 4 distinct ticket scenarios via `orchestrator.invoke` with classification, routing, tool usage, resolution, automatic escalation, and redacted structured event logging.

## 1. Setup & Environment

In [1]:
import sys, os
PROJECT_ROOT = os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from unittest.mock import patch
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.outputs import ChatResult, ChatGeneration
from agentic.workflow import orchestrator, classify_ticket
from agentic.logger import log_event, get_ticket_events, get_metrics_summary, clear_logs
from agentic.tools import get_ticket_details

load_dotenv()
clear_logs()
print("[OK] Environment initialized successfully.")

[OK] Environment initialized successfully.


## 2. Ticket Scenario 1: Policy / FAQ Query (Successful Resolution)

- **Query**: How to cancel or pause subscription?
- **Workflow**: `orchestrator.invoke` -> Classified as `policy` -> Routed to `support_agent` -> RAG search -> Resolved in DB.

In [1]:
ticket_id_1 = "demo_ticket_policy_001"
input_1 = {
    "messages": [HumanMessage(content="How do I cancel or pause my CultPass subscription?")],
    "ticket_metadata": {"ticket_id": ticket_id_1, "tags": "cancellation, policy", "urgency": "normal"}
}
config_1 = {"configurable": {"thread_id": ticket_id_1}}

tool_call_msg_1 = AIMessage(
    content="",
    tool_calls=[{"name": "search_knowledge_base", "args": {"query": "how to cancel or pause subscription"}, "id": "call_1"}]
)
final_ans_1 = AIMessage(content="You can cancel or pause your subscription at any time via the My Account section in the CultPass app.")

res1_mock = ChatResult(generations=[ChatGeneration(message=tool_call_msg_1)])
res2_mock = ChatResult(generations=[ChatGeneration(message=final_ans_1)])

with patch("langchain_openai.ChatOpenAI._generate", side_effect=[res1_mock, res2_mock]):
    res_1 = orchestrator.invoke(input_1, config=config_1)
    print(f"[ASSISTANT RESPONSE]: {res_1['messages'][-1].content}")

events_1 = get_ticket_events(ticket_id_1)
print(f"[STRUCTURED EVENTS]: Recorded {len(events_1)} events: {[e['event_type'] for e in events_1]}")

[ASSISTANT RESPONSE]: You can cancel or pause your subscription at any time via the My Account section in the CultPass app.
[STRUCTURED EVENTS]: Recorded 7 events: ['CLASSIFICATION', 'ROUTING', 'AGENT_EXECUTION', 'TOOL_CALL', 'RETRIEVAL_SUCCESS', 'AGENT_EXECUTION', 'RESOLUTION']


## 3. Ticket Scenario 2: Low-Confidence Knowledge -> Automatic Escalation Handoff

- **Query**: Inquiry regarding non-existent feature.
- **Workflow**: `orchestrator.invoke` -> Classified -> RAG search returns `should_escalate: True` -> Automatic Escalation in DB.

In [1]:
ticket_id_2 = "demo_ticket_escalate_002"
input_2 = {
    "messages": [HumanMessage(content="I need help with xyz123999 unknown quantum portal feature.")],
    "ticket_metadata": {"ticket_id": ticket_id_2, "tags": "unknown, bug", "urgency": "high"}
}
config_2 = {"configurable": {"thread_id": ticket_id_2}}

tool_call_msg_2 = AIMessage(
    content="",
    tool_calls=[{"name": "search_knowledge_base", "args": {"query": "xyz123999 unknown quantum portal feature"}, "id": "call_2"}]
)
res_mock_2 = ChatResult(generations=[ChatGeneration(message=tool_call_msg_2)])

with patch("langchain_openai.ChatOpenAI._generate", return_value=res_mock_2):
    res_2 = orchestrator.invoke(input_2, config=config_2)
    print(f"[AUTOMATIC ESCALATION HANDOFF]: Ticket escalated to human support lead.")

events_2 = get_ticket_events(ticket_id_2)
print(f"[ESCALATION EVENTS]: Recorded {[e['event_type'] for e in events_2 if e['event_type'] in ['RETRIEVAL_MISS', 'ESCALATION']]}")

[AUTOMATIC ESCALATION HANDOFF]: Ticket escalated to human support lead.
[ESCALATION EVENTS]: Recorded ['RETRIEVAL_MISS', 'ESCALATION']


## 4. Ticket Scenario 3: Account & Reservation Services

- **Query**: Look up user profile and active reservations for `a4ab87`.
- **Workflow**: `orchestrator.invoke` -> Classified as `account` -> Routed to `account_agent` -> Profile & Reservations fetched -> Resolved.

In [1]:
ticket_id_3 = "demo_ticket_account_003"
input_3 = {
    "messages": [HumanMessage(content="Please check pass quota and active reservations for user a4ab87.")],
    "ticket_metadata": {"ticket_id": ticket_id_3, "tags": "profile, quota", "urgency": "normal"}
}
config_3 = {"configurable": {"thread_id": ticket_id_3}}

tool_call_msg_3 = AIMessage(
    content="",
    tool_calls=[{"name": "get_user_profile", "args": {"user_id_or_email": "a4ab87"}, "id": "call_3"}]
)
final_ans_3 = AIMessage(content="User a4ab87 has an active basic subscription.")

res1_mock_3 = ChatResult(generations=[ChatGeneration(message=tool_call_msg_3)])
res2_mock_3 = ChatResult(generations=[ChatGeneration(message=final_ans_3)])

with patch("langchain_openai.ChatOpenAI._generate", side_effect=[res1_mock_3, res2_mock_3]):
    res_3 = orchestrator.invoke(input_3, config=config_3)
    print(f"[ASSISTANT RESPONSE]: {res_3['messages'][-1].content}")

events_3 = get_ticket_events(ticket_id_3)
tool_calls_3 = [e.get('tool_name') for e in events_3 if e.get('event_type') == 'TOOL_CALL']
print(f"[ACCOUNT TOOL CALLS]: {tool_calls_3}")

[ASSISTANT RESPONSE]: User a4ab87 has an active basic subscription.
[ACCOUNT TOOL CALLS]: ['get_user_profile']


## 5. Ticket Scenario 4: Error Handling & Edge Case

- **Query**: Reservation lookup with non-existent user ID.
- **Workflow**: `orchestrator.invoke` -> Tool validation catches unknown user -> Structured error returned -> Handled cleanly.

In [1]:
ticket_id_4 = "demo_ticket_edge_004"
input_4 = {
    "messages": [HumanMessage(content="Look up reservations for nonexistent_user_xyz999.")],
    "ticket_metadata": {"ticket_id": ticket_id_4, "tags": "account", "urgency": "normal"}
}
config_4 = {"configurable": {"thread_id": ticket_id_4}}

tool_call_msg_4 = AIMessage(
    content="",
    tool_calls=[{"name": "get_user_reservations", "args": {"user_id": "nonexistent_user_xyz999"}, "id": "call_4"}]
)
final_ans_4 = AIMessage(content="User nonexistent_user_xyz999 was not found.")

res1_mock_4 = ChatResult(generations=[ChatGeneration(message=tool_call_msg_4)])
res2_mock_4 = ChatResult(generations=[ChatGeneration(message=final_ans_4)])

with patch("langchain_openai.ChatOpenAI._generate", side_effect=[res1_mock_4, res2_mock_4]):
    res_4 = orchestrator.invoke(input_4, config=config_4)
    print(f"[EDGE CASE RESPONSE]: {res_4['messages'][-1].content}")

events_4 = get_ticket_events(ticket_id_4)
error_events = [e for e in events_4 if e.get('outcome') == 'error']
print(f"[ERROR EVENTS LOGGED]: {len(error_events)} error events recorded.")

[EDGE CASE RESPONSE]: User nonexistent_user_xyz999 was not found.
[ERROR EVENTS LOGGED]: 1 error events recorded.


## 6. Structured Operational Metrics & Redacted Log Trace Summary

In [1]:
metrics = get_metrics_summary()
print("=" * 60)
print("            STRUCTURED OPERATIONAL METRICS SUMMARY          ")
print("=" * 60)
print(f"Total Events Recorded     : {metrics['total_events']}")
print(f"Unique Tickets Processed  : {metrics['unique_tickets']}")
print(f"Total Tool Calls Executed : {metrics['total_tool_calls']}")
print(f"Retrieval Success Rate    : {metrics['retrieval_success_rate'] * 100:.1f}%")
print(f"Human Escalation Count   : {metrics['escalation_count']}")
print(f"Tool Usage Breakdown     : {metrics['tool_usage_counts']}")

            STRUCTURED OPERATIONAL METRICS SUMMARY          
Total Events Recorded     : 27
Unique Tickets Processed  : 4
Total Tool Calls Executed : 4
Retrieval Success Rate    : 150.0%
Human Escalation Count   : 1
Tool Usage Breakdown     : {'search_knowledge_base': 2, 'get_user_profile': 1, 'get_user_reservations': 1}
